# **Task 1**

In [1]:
import json
import math
import os
import time
import urllib.request

import torch
import torch.nn as nn
from torch.nn import functional as F

In [2]:
SEED = 1337
BATCH_SIZE = 32        # aynı anda işlenen bağımsız dizi sayısı
BLOCK_SIZE = 8         # bir chunk'taki maksimum context uzunluğu
MAX_ITERS = 3000
EVAL_INTERVAL = 300
EVAL_ITERS = 200       # train/val loss tahmini için ortalaması alınan batch sayısı
LEARNING_RATE = 1e-2
TRAIN_FRAC = 0.9

DATA_URL = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
HERE = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()
DATA_PATH = os.path.join(HERE, 'input.txt')
RESULTS_PATH = os.path.join(HERE, 'bigram_baseline.json')

if torch.cuda.is_available():
    DEVICE = 'cuda'
elif getattr(torch.backends, 'mps', None) is not None and torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'

In [3]:
def load_text(path=DATA_PATH, url=DATA_URL):
    """Tiny Shakespeare'i oku; dosya yoksa önce indir."""
    if not os.path.exists(path):
        print(f'{os.path.basename(path)} bulunamadı, indiriliyor...')
        urllib.request.urlretrieve(url, path)
    with open(path, 'r', encoding='utf-8') as f:
        return f.read()


In [10]:
def build_tokenizer(text):
    """Metindeki her farklı karakter bir token.

    Hafta 3'teki isim verisinde her ismin başına/sonuna '.' koyup ona 0 veriyorduk,
    çünkü isimler birbirinden bağımsızdı. Burada metin tek parça bir akış;
    satır sonu zaten metnin kendi karakteri, ayrıca özel token gerekmiyor.
    """
    chars = sorted(set(text))
    stoi = {ch: i for i, ch in enumerate(chars)}
    itos = {i: ch for ch, i in stoi.items()}

    def encode(s):
        return [stoi[c] for c in s]          # string -> list[int]

    def decode(ids):
        return ''.join(itos[i] for i in ids)  # list[int] -> string

    return chars, encode, decode

In [5]:
def train_val_split(data, train_frac=TRAIN_FRAC):
    """İlk %90 train, son %10 val.

    Hafta 4'te isimleri bölmeden önce karıştırıyorduk; burada karıştırmıyoruz,
    çünkü karakterleri karıştırmak metni yok eder. Val, modelin hiç görmediği
    ardışık bir metin parçası oluyor.
    """
    n = int(train_frac * len(data))
    return data[:n], data[n:]

In [6]:
def get_batch(data, batch_size=BATCH_SIZE, block_size=BLOCK_SIZE, device=DEVICE):
    """Veriden rastgele batch_size tane chunk çek.

    x: (B, T)  girdiler
    y: (B, T)  hedefler = x'in bir karakter sağa kaydırılmışı
    Her chunk aslında T tane örnek taşır: x[:t+1] bağlamından y[t] tahmin edilir.
    """
    ix = torch.randint(len(data) - block_size, (batch_size,))   # chunk başlangıçları
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

In [7]:
class BigramLanguageModel(nn.Module):
    """Her token, bir sonraki tokenın logit'lerini (V x V) tablodan doğrudan okur.

    Bağlam yok: tahmin sadece son karaktere bakar
    """

    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)  # init: N(0,1) = torch.randn

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)            # (B, T, V)
        loss = None
        if targets is not None:
            B, T, V = logits.shape
            # cross_entropy (N, V) logits ve (N,) hedef bekler -> B ve T'yi birleştir
            loss = F.cross_entropy(logits.view(B * T, V), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        """idx: (B, T) başlangıç bağlamı. Her adımda bir karakter örnekleyip ekler."""
        for _ in range(max_new_tokens):
            logits, _ = self(idx)
            logits = logits[:, -1, :]                        # sadece son zaman adımı -> (B, V)
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)   # argmax değil, sample (Hafta 3)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

In [8]:
# Değerlendirme yardımcıları (sonraki görevlerde GPT için de kullanılacak)
# ---------------------------------------------------------------------------
@torch.no_grad()
def estimate_loss(model, splits, eval_iters=EVAL_ITERS, batch_size=BATCH_SIZE,
                  block_size=BLOCK_SIZE, device=DEVICE):
    """Her split için eval_iters rastgele batch'in ortalama loss'u (videodaki yöntem)."""
    model.eval()
    out = {}
    for name, data in splits.items():
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(data, batch_size, block_size, device)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[name] = losses.mean().item()
    model.train()
    return out


@torch.no_grad()
def full_split_loss(model, data, block_size=BLOCK_SIZE, device=DEVICE, batch_size=1024):
    """Split'in tamamı üzerinden kesin (rastgelelik içermeyen) ortalama loss.

    Veri, örtüşmeyen block_size uzunluğunda chunk'lara bölünür ve her pozisyonun
    loss'u toplanır. Rastgele batch tahmininden farklı olarak her çalıştırmada aynı sonucu verir.
    """
    model.eval()
    n_chunks = (len(data) - 1) // block_size
    x = data[:n_chunks * block_size].view(n_chunks, block_size)
    y = data[1:n_chunks * block_size + 1].view(n_chunks, block_size)
    total, count = 0.0, 0
    for i in range(0, n_chunks, batch_size):
        xb, yb = x[i:i + batch_size].to(device), y[i:i + batch_size].to(device)
        logits, _ = model(xb)
        total += F.cross_entropy(logits.reshape(-1, logits.size(-1)), yb.reshape(-1),
                                 reduction='sum').item()
        count += yb.numel()
    model.train()
    return total / count


def count_bigram_nll(train_data, eval_data, vocab_size, smoothing=1.0):
    """Hafta 3'ün sayım modeli: train'deki bigram'ları say, +1 smoothing, satır normalize et.

    Bu, train verisinden öğrenilebilecek en iyi bigram tablosunun val'deki skoru;
    gradient'le eğitilen nn.Module'ün hedefi. Ona yaklaşıyorsa doğru yakınsamış demektir.
    """
    N = torch.zeros((vocab_size, vocab_size))
    N.index_put_((train_data[:-1], train_data[1:]), torch.ones(len(train_data) - 1), accumulate=True)
    P = (N + smoothing) / (N + smoothing).sum(1, keepdim=True)   # keepdim! (Hafta 3, Task 2)
    return -P[eval_data[:-1], eval_data[1:]].log().mean().item()


def sanity_checks(model, xb, yb, vocab_size):
    """Hafta 3'teki elle yazılmış hesaplarla nn.Module sürümünün aynı olduğunu doğrula."""
    logits, loss = model(xb, yb)
    W = model.token_embedding_table.weight                          # (V, V) = Hafta 3'teki W

    # (a) one_hot(x) @ W == embedding lookup
    xenc = F.one_hot(xb, num_classes=vocab_size).float()             # (B, T, V)
    assert torch.allclose(xenc @ W, logits, atol=1e-6), 'one-hot @ W ile embedding farklı!'

    # (b) elle softmax + NLL == F.cross_entropy
    flat_logits, flat_y = logits.view(-1, vocab_size), yb.view(-1)
    counts = flat_logits.exp()
    probs = counts / counts.sum(1, keepdim=True)
    manual_nll = -probs[torch.arange(flat_y.numel()), flat_y].log().mean()
    assert torch.allclose(manual_nll, loss, atol=1e-5), 'elle NLL ile cross_entropy farklı!'
    return loss.item()


In [9]:
def main():
    torch.manual_seed(SEED)
    t0 = time.time()

    # 1. veri
    text = load_text()
    print(f'karakter sayısı: {len(text):,}')

    # 2. tokenizer
    chars, encode, decode = build_tokenizer(text)
    vocab_size = len(chars)
    print(f'vocab_size: {vocab_size}  ->  {"".join(chars)!r}')
    print(f'encode("hii there") = {encode("hii there")}')
    assert decode(encode(text)) == text, 'encode/decode tüm metinde geri dönüşümlü değil!'

    data = torch.tensor(encode(text), dtype=torch.long)

    # 3. split
    train_data, val_data = train_val_split(data)
    assert len(train_data) + len(val_data) == len(data)
    print(f'train: {len(train_data):,} token | val: {len(val_data):,} token')

    # 4. get_batch
    xb, yb = get_batch(train_data, batch_size=4)
    assert xb.shape == yb.shape == (4, BLOCK_SIZE)
    print(f'\nörnek batch: x {tuple(xb.shape)}, y {tuple(yb.shape)}')
    for t in range(BLOCK_SIZE):   # ilk chunk'taki gizli örnekler
        print(f'  bağlam {decode(xb[0, :t + 1].tolist())!r:12} -> hedef {decode([yb[0, t].item()])!r}')

    # 5. model
    model = BigramLanguageModel(vocab_size).to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters())
    init_loss = sanity_checks(model, *get_batch(train_data), vocab_size)
    print(f'\nparametre: {n_params:,} (= {vocab_size} x {vocab_size})')
    print(f'eğitim öncesi loss: {init_loss:.4f}  (uniform tahmin: -ln(1/{vocab_size}) = {math.log(vocab_size):.4f})')

    # Hafta 3: full-batch gradient descent, W.data += -50 * W.grad.
    # Burada mini-batch + AdamW. AdamW'nin varsayılan weight_decay=0.01'i, Hafta 3'teki
    # 0.01 * (W**2).mean() regularization'ının akrabası: ağırlıkları her adımda 0'a doğru çeker.
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    splits = {'train': train_data, 'val': val_data}
    history = []

    print(f'\neğitim ({DEVICE}):')
    for it in range(MAX_ITERS):
        if it % EVAL_INTERVAL == 0 or it == MAX_ITERS - 1:
            losses = estimate_loss(model, splits)
            history.append({'step': it, **losses})
            print(f'  step {it:5d}: train {losses["train"]:.4f} | val {losses["val"]:.4f}')

        X, Y = get_batch(train_data)
        _, loss = model(X, Y)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

    # final değerlendirme
    final = history[-1]
    full_train = full_split_loss(model, train_data)
    full_val = full_split_loss(model, val_data)
    count_val = count_bigram_nll(train_data, val_data, vocab_size)
    count_train = count_bigram_nll(train_data, train_data, vocab_size)

    print('\nsonuç:')
    print(f'  val loss (tahmini, {EVAL_ITERS} batch): {final["val"]:.4f}')
    print(f'  val loss (tüm val seti, kesin):     {full_val:.4f}   <- karşılaştırma tabanı')
    print(f'  train loss (tüm train seti):        {full_train:.4f}')
    print(f'  sayım modeli val NLL (referans):    {count_val:.4f}   fark: {full_val - count_val:+.4f}')

    context = torch.zeros((1, 1), dtype=torch.long, device=DEVICE)   # 0 = '\n'
    sample = decode(model.generate(context, max_new_tokens=500)[0].tolist())
    print('\nörnek üretim:\n' + sample)

    results = {
        'model': 'bigram',
        'val_loss': round(full_val, 4),
        'val_loss_estimate': round(final['val'], 4),
        'train_loss': round(full_train, 4),
        'train_loss_estimate': round(final['train'], 4),
        'count_model_val_nll': round(count_val, 4),
        'count_model_train_nll': round(count_train, 4),
        'initial_loss': round(init_loss, 4),
        'uniform_loss': round(math.log(vocab_size), 4),
        'n_params': n_params,
        'vocab_size': vocab_size,
        'n_train_tokens': len(train_data),
        'n_val_tokens': len(val_data),
        'config': {
            'seed': SEED, 'batch_size': BATCH_SIZE, 'block_size': BLOCK_SIZE,
            'max_iters': MAX_ITERS, 'eval_interval': EVAL_INTERVAL, 'eval_iters': EVAL_ITERS,
            'learning_rate': LEARNING_RATE, 'optimizer': 'AdamW', 'train_frac': TRAIN_FRAC,
        },
        'history': [{k: (round(v, 4) if isinstance(v, float) else v) for k, v in h.items()} for h in history],
        'sample': sample,
        'device': DEVICE,
        'torch_version': torch.__version__,
        'runtime_sec': round(time.time() - t0, 1),
    }
    with open(RESULTS_PATH, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    print(f'\nkaydedildi: {RESULTS_PATH}')
    return results


if __name__ == '__main__':
    main()

input.txt bulunamadı, indiriliyor...
karakter sayısı: 1,115,394
vocab_size: 65  ->  "\n !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz"
encode("hii there") = [46, 47, 47, 1, 58, 46, 43, 56, 43]
train: 1,003,854 token | val: 111,540 token

örnek batch: x (4, 8), y (4, 8)
  bağlam 'L'          -> hedef 'e'
  bağlam 'Le'         -> hedef 't'
  bağlam 'Let'        -> hedef "'"
  bağlam "Let'"       -> hedef 's'
  bağlam "Let's"      -> hedef ' '
  bağlam "Let's "     -> hedef 'h'
  bağlam "Let's h"    -> hedef 'e'
  bağlam "Let's he"   -> hedef 'a'

parametre: 4,225 (= 65 x 65)
eğitim öncesi loss: 4.6816  (uniform tahmin: -ln(1/65) = 4.1744)

eğitim (cpu):
  step     0: train 4.6352 | val 4.6489
  step   300: train 2.8162 | val 2.8343
  step   600: train 2.5439 | val 2.5779
  step   900: train 2.4925 | val 2.5144
  step  1200: train 2.4853 | val 2.5072
  step  1500: train 2.4664 | val 2.4957
  step  1800: train 2.4682 | val 2.4961
  step  2100: train 2.4701 | val 2.4875
  

# **Task 2**

In [24]:
import time
import torch
from torch.nn import functional as F

In [25]:
torch.manual_seed(1337)
B, T, C = 4, 8, 2          # batch, time (context uzunluğu), channels
x = torch.randn(B, T, C)
print('x:', tuple(x.shape))

x: (4, 8, 2)


In [26]:
torch.manual_seed(42)
a = torch.tril(torch.ones(3, 3))
a = a / a.sum(1, keepdim=True)          # her satırı kendi toplamına böl
b = torch.randint(0, 10, (3, 2)).float()
c = a @ b
print('a (ağırlıklar):\n', a)
print('b (değerler):\n', b)
print('c = a @ b:\n', c)
print()
print('c[0] = b[0]                  ->', b[0].tolist())
print('c[1] = (b[0] + b[1]) / 2     ->', ((b[0] + b[1]) / 2).tolist())
print('c[2] = (b[0] + b[1] + b[2])/3 ->', ((b[0] + b[1] + b[2]) / 3).tolist())
assert torch.allclose(c, torch.stack([b[:1].mean(0), b[:2].mean(0), b[:3].mean(0)]))

a (ağırlıklar):
 tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
b (değerler):
 tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
c = a @ b:
 tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])

c[0] = b[0]                  -> [2.0, 7.0]
c[1] = (b[0] + b[1]) / 2     -> [4.0, 5.5]
c[2] = (b[0] + b[1] + b[2])/3 -> [4.666666507720947, 5.333333492279053]


In [27]:
## 1. Yol: for döngüsü (42:13)
# En açık yazım: her batch ve her t için 0..t dilimini al, ortalamasını hesapla. !!!!!(bu yönetmin eksileri yavaş ve her değere eşit bir weight veriyor)!!!!!

# %%
xbow1 = torch.zeros((B, T, C))
for b_ in range(B):
    for t in range(T):
        xprev = x[b_, :t + 1]               # (t+1, C): geçmiş + kendisi
        xbow1[b_, t] = torch.mean(xprev, 0)
print('xbow1:', tuple(xbow1.shape))
print('x[0]:\n', x[0])
print('xbow1[0] (her satır, x[0]\'ın o satıra kadarki ortalaması):\n', xbow1[0])

xbow1: (4, 8, 2)
x[0]:
 tensor([[ 0.1808, -0.0700],
        [-0.3596, -0.9152],
        [ 0.6258,  0.0255],
        [ 0.9545,  0.0643],
        [ 0.3612,  1.1679],
        [-1.3499, -0.5102],
        [ 0.2360, -0.2398],
        [-0.9211,  1.5433]])
xbow1[0] (her satır, x[0]'ın o satıra kadarki ortalaması):
 tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])


In [15]:
## 2. Yol: torch.tril ile matris çarpımı (51:54)
# Alt üçgen matrisin her satırını kendi toplamına bölünce t'inci satır
# 1/(t+1) değerlerinden oluşur: ilk t+1 elemana eşit ağırlık, gerisine 0.
# `keepdim=True`: Hafta 3, Task 2'deki tuzak. Satır toplamı (T, 1) kalmalı ki bölme satır bazında yayılsın.

# %%
wei2 = torch.tril(torch.ones(T, T))
wei2 = wei2 / wei2.sum(1, keepdim=True)
xbow2 = wei2 @ x                            # (T, T) @ (B, T, C) -> (B, T, T) @ (B, T, C) -> (B, T, C)
print('wei2:\n', wei2)
print('xbow2:', tuple(xbow2.shape))

wei2:
 tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])
xbow2: (4, 8, 2)


In [28]:
## 3. Yol: softmax ile

tril = torch.tril(torch.ones(T, T))
wei3 = torch.zeros((T, T))                          # skorlar (affinities), şimdilik hepsi eşit.  e üzeri eksi sonsuz 0 , e üzeri 0  1 eder.
wei3 = wei3.masked_fill(tril == 0, float('-inf'))   # gelecek: -inf
print('softmax öncesi skorlar:\n', wei3)
wei3 = F.softmax(wei3, dim=-1)                      # exp(-inf)=0, exp(0)=1 -> satır normalize
xbow3 = wei3 @ x
print('softmax sonrası ağırlıklar:\n', wei3)

softmax öncesi skorlar:
 tensor([[0., -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0., 0., 0., 0.]])
softmax sonrası ağırlıklar:
 tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.

In [29]:
## Karşılaştırma: üçü de aynı mı?
# float32'de döngü ile matris çarpımı toplamayı farklı sırada yapar; sonuçlar son basamakta
# (~1e-8) farklılaşabilir. `torch.allclose`'un varsayılan toleransı `atol=1e-8, rtol=1e-5`;
# sıfıra çok yakın bir elemanda bu kadar küçük bir fark bile eşiği aşabilir.
# Bu yüzden float32 hassasiyetine uygun `atol=1e-6` kullanıyoruz ve aynı karşılaştırmayı
# float64'te varsayılan toleransla da tekrarlıyoruz.

# %%
ATOL = 1e-6
print('ağırlıklar aynı  (wei2 == wei3)  :', torch.allclose(wei2, wei3))
print('döngü  == tril   (xbow1 == xbow2):', torch.allclose(xbow1, xbow2, atol=ATOL))
print('döngü  == softmax(xbow1 == xbow3):', torch.allclose(xbow1, xbow3, atol=ATOL))
print('tril   == softmax(xbow2 == xbow3):', torch.allclose(xbow2, xbow3, atol=ATOL))
print(f'en büyük mutlak fark: {max((xbow1 - xbow2).abs().max(), (xbow1 - xbow3).abs().max()).item():.2e}'
      f'   (float32 hassasiyeti: {torch.finfo(torch.float32).eps:.2e})')
assert torch.allclose(xbow1, xbow2, atol=ATOL) and torch.allclose(xbow1, xbow3, atol=ATOL)

# float64'te aynı üç yol, varsayılan toleransla
x64 = x.double()
xbow1_64 = torch.zeros((B, T, C), dtype=torch.float64)
for b_ in range(B):
    for t in range(T):
        xbow1_64[b_, t] = x64[b_, :t + 1].mean(0)
xbow2_64 = wei2.double() @ x64
xbow3_64 = wei3.double() @ x64
print('float64, varsayılan tolerans  -> döngü == tril:', torch.allclose(xbow1_64, xbow2_64),
      '| döngü == softmax:', torch.allclose(xbow1_64, xbow3_64))
assert torch.allclose(xbow1_64, xbow2_64) and torch.allclose(xbow1_64, xbow3_64)


ağırlıklar aynı  (wei2 == wei3)  : True
döngü  == tril   (xbow1 == xbow2): True
döngü  == softmax(xbow1 == xbow3): True
tril   == softmax(xbow2 == xbow3): True
en büyük mutlak fark: 3.24e-08   (float32 hassasiyeti: 1.19e-07)
float64, varsayılan tolerans  -> döngü == tril: True | döngü == softmax: True


In [18]:
## Matris çarpımı neden ağırlıklı ortalama?


#
t = 3
elle = sum(wei2[t, s] * x[0, s] for s in range(T))   # sum_s wei[t, s] * x[s]
print(f'wei2[{t}] =', [round(w, 3) for w in wei2[t].tolist()])
print(f'elle toplam   : {elle.tolist()}')
print(f'matris çarpımı: {xbow2[0, t].tolist()}')
assert torch.allclose(elle, xbow2[0, t])

wei2[3] = [0.25, 0.25, 0.25, 0.25, 0.0, 0.0, 0.0, 0.0]
elle toplam   : [0.3503567576408386, -0.22383351624011993]
matris çarpımı: [0.3503567576408386, -0.22383351624011993]


In [20]:
#Ağırlıklar eşit olmak zorunda değil. Skorları rastgele seçince softmax yine
# negatif olmayan ve toplamı 1 olan ağırlıklar üretir. Sonuç düz ortalama değil,
# **ağırlıklı** ortalama olur. Self-attention'da bu skorlar sabit 0 yerine
# query · key'den gelecek.

# %%
torch.manual_seed(7)
skor = torch.randn(T, T).masked_fill(tril == 0, float('-inf'))
wei_rnd = F.softmax(skor, dim=-1)
out_rnd = wei_rnd @ x
print('rastgele skorlarla ağırlıklar:\n', wei_rnd.round(decimals=3))
print('satır toplamları:', wei_rnd.sum(1).round(decimals=4).tolist())
# ağırlıklı ortalama, geçmiş değerlerin en küçüğü ile en büyüğü arasında kalmak zorunda
gecmis = [x[:, :i + 1] for i in range(T)]
icinde = all(((out_rnd[:, i] >= g.min(1).values - 1e-6) & (out_rnd[:, i] <= g.max(1).values + 1e-6)).all()
             for i, g in enumerate(gecmis))
print('her çıktı, geçmişin min-max aralığında mı:', icinde)


rastgele skorlarla ağırlıklar:
 tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.8120, 0.1880, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0260, 0.0930, 0.8810, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3260, 0.3240, 0.0650, 0.2850, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0590, 0.0620, 0.0760, 0.6360, 0.1660, 0.0000, 0.0000, 0.0000],
        [0.0220, 0.3340, 0.4710, 0.0320, 0.0700, 0.0710, 0.0000, 0.0000],
        [0.1900, 0.1750, 0.2240, 0.1550, 0.0120, 0.0790, 0.1670, 0.0000],
        [0.3760, 0.0740, 0.0480, 0.0090, 0.2330, 0.0510, 0.0520, 0.1580]])
satır toplamları: [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
her çıktı, geçmişin min-max aralığında mı: True


In [30]:
 ## Neden uğraşıyoruz: hız
# Aynı ortalamayı gerçekçi boyutlarda (B=32, T=256, C=64) iki yoldan hesaplayalım.


torch.manual_seed(0)
Bb, Tb, Cb = 32, 256, 64
xb = torch.randn(Bb, Tb, Cb)

t0 = time.perf_counter()
loop = torch.zeros_like(xb)
for b_ in range(Bb):
    for t in range(Tb):
        loop[b_, t] = xb[b_, :t + 1].mean(0)
t_loop = time.perf_counter() - t0

t0 = time.perf_counter()
w = torch.tril(torch.ones(Tb, Tb))
mm = (w / w.sum(1, keepdim=True)) @ xb
t_mm = time.perf_counter() - t0

print(f'for döngüsü    : {t_loop * 1000:8.1f} ms')
print(f'matris çarpımı : {t_mm * 1000:8.1f} ms   (~{t_loop / t_mm:.0f}x hızlı)')
print('aynı sonuç     :', torch.allclose(loop, mm, atol=1e-6))




for döngüsü    :    224.9 ms
matris çarpımı :      5.1 ms   (~44x hızlı)
aynı sonuç     : True


# **Task 3 **

In [46]:
import os
import urllib.request

import torch
import torch.nn as nn
from torch.nn import functional as F

In [47]:
torch.manual_seed(1337)

# --- veri ve tokenizer (Görev 1 ile aynı) ---
if not os.path.exists('input.txt'):
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt',
        'input.txt')
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join(itos[i] for i in l)
data = torch.tensor(encode(text), dtype=torch.long)

# --- boyutlar (videodaki örnekle aynı) ---
B, T = 4, 8          # 4 dizi, her biri 8 karakter (block_size)
n_embd = 32          # C: her token'ın vektör boyutu
head_size = 16       # query/key/value boyutu


def goster(ch):
    """Boşluk ve satır sonu çıktıda görünsün diye."""
    return {' ': '␣', '\n': '⏎'}.get(ch, ch)

In [48]:
## 1. Token + pozisyon embedding'i
# Token embedding: "hangi karakter?"  Pozisyon embedding: "dizinin kaçıncı yerinde?"
# İkisi toplanınca x her token için hem kimliği hem yeri taşır.

# %%
print('=== 1. TOKEN + POZİSYON EMBEDDING ===')
offsets = [0, 9, 100, 1000]                                  # sabit örnekler, her çalıştırmada aynı
idx = torch.stack([data[i:i + T] for i in offsets])          # (B, T)
for b, i in enumerate(offsets):
    print(f'dizi {b}: {decode(idx[b].tolist())!r}')

token_embedding_table = nn.Embedding(vocab_size, n_embd)     # (65, 32)
position_embedding_table = nn.Embedding(T, n_embd)           # (8, 32): her pozisyon için bir vektör

tok_emb = token_embedding_table(idx)                         # (B, T, C)
pos_emb = position_embedding_table(torch.arange(T))          # (T, C) -> B boyunca broadcast
x = tok_emb + pos_emb                                        # (B, T, C)
print('\ntok_emb', tuple(tok_emb.shape), '+ pos_emb', tuple(pos_emb.shape), '-> x', tuple(x.shape))


=== 1. TOKEN + POZİSYON EMBEDDING ===
dizi 0: 'First Ci'
dizi 1: 'izen:\nBe'
dizi 2: ' are all'
dizi 3: 'Second C'

tok_emb (4, 8, 32) + pos_emb (8, 32) -> x (4, 8, 32)


In [49]:
## 2. Tek bir self-attention head
# Her token üç vektör üretir:
# - query : "ne arıyorum?"
# - key   : "bende ne var?"
# - value : "bana dikkat edersen sana bunu veririm"
# Skor (affinity) = query · key. Bir token'ın query'si başka bir token'ın key'iyle
# ne kadar örtüşürse ona o kadar dikkat eder. Görev 2'deki sabit 0 skorların yerini bu alır.

# %%
class Head(nn.Module):
    """Tek bir self-attention head (decoder: gelecek maskeli)."""

    def __init__(self, n_embd, head_size, block_size):
        super().__init__()
        self.head_size = head_size
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        # tril bir parametre değil, eğitilmez -> buffer
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.wei = None   # son hesaplanan ağırlıklar, incelemek için

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)                                              # (B, T, hs)
        q = self.query(x)                                            # (B, T, hs)
        # Videodaki düzeltme (1:20:05): C ile değil head_size ile ölçekle
        wei = q @ k.transpose(-2, -1) * self.head_size ** -0.5       # (B, T, hs) @ (B, hs, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))  # gelecek görünmesin
        wei = F.softmax(wei, dim=-1)                                 # her satır toplamı 1
        self.wei = wei.detach()
        v = self.value(x)                                            # (B, T, hs)
        out = wei @ v                                                # (B, T, T) @ (B, T, hs) -> (B, T, hs)
        return out


print('\n=== 2. SELF-ATTENTION HEAD ===')
head = Head(n_embd, head_size, block_size=T)
out = head(x)
wei = head.wei
print('wei:', tuple(wei.shape), '| out:', tuple(out.shape))
print('Görev 2\'de wei (T, T) idi, herkes aynı ağırlıkları kullanıyordu.')
print('Şimdi (B, T, T): her dizinin ağırlıkları kendi verisinden hesaplanıyor.')



=== 2. SELF-ATTENTION HEAD ===
wei: (4, 8, 8) | out: (4, 8, 16)
Görev 2'de wei (T, T) idi, herkes aynı ağırlıkları kullanıyordu.
Şimdi (B, T, T): her dizinin ağırlıkları kendi verisinden hesaplanıyor.


In [50]:
## 3. Ağırlık matrisi ve bir satırının okunması

# %%
etiket = [goster(itos[i]) for i in idx[0].tolist()]


def matris_yazdir(w, etiket):
    print('satır = bakan token, sütun = bakılan token\n')
    print(' ' * 7 + ''.join(f'{f"{j}:{e}":>7}' for j, e in enumerate(etiket)))
    for i in range(len(etiket)):
        hucre = ''.join(f'{w[i, j].item():7.3f}' if j <= i else f'{"·":>7}' for j in range(len(etiket)))
        print(f'{f"{i}:{etiket[i]}":>5}  ' + hucre)


print('\n=== 3. AĞIRLIK MATRİSİ (wei) ===')
print(f'dizi 0 = {decode(idx[0].tolist())!r}\n')
matris_yazdir(wei[0], etiket)
print('\nsatır toplamları:', [round(s, 4) for s in wei[0].sum(-1).tolist()])

t = T - 1
satir = wei[0, t]
print(f"\nSatır t={t}: {etiket[t]!r} token'ı (pozisyon {t}) geçmişte kime ne kadar bakıyor?")
for s in satir[:t + 1].argsort(descending=True).tolist():
    print(f"  pozisyon {s} {etiket[s]!r:5} {satir[s].item():6.1%}  {'█' * round(satir[s].item() * 60)}")
en = satir.argmax().item()
print(f"-> En çok baktığı: pozisyon {en}, {etiket[en]!r} ({satir[en].item():.1%}).")
print(f"   Görev 2'deki düz ortalamada herkes eşit, 1/{t + 1} = {1 / (t + 1):.1%} alırdı.")
print('   Head eğitilmediği (ağırlıklar rastgele) için bu tercih henüz dil açısından anlamlı değil.')
print('   Eğitimle query/key öğrenildikçe bu satır "tahmin için hangi geçmiş karakter işime yarar?"')
print('   sorusunun cevabına dönüşür.')

# aynı pozisyon, farklı dizi -> farklı ağırlıklar
print(f'\nAynı satır (t={t}), farklı diziler:')
for b in range(2):
    print(f'  dizi {b} {decode(idx[b].tolist())!r}:', [round(v, 3) for v in wei[b, t].tolist()])



=== 3. AĞIRLIK MATRİSİ (wei) ===
dizi 0 = 'First Ci'

satır = bakan token, sütun = bakılan token

           0:F    1:i    2:r    3:s    4:t    5:␣    6:C    7:i
  0:F    1.000      ·      ·      ·      ·      ·      ·      ·
  1:i    0.584  0.416      ·      ·      ·      ·      ·      ·
  2:r    0.409  0.202  0.389      ·      ·      ·      ·      ·
  3:s    0.471  0.194  0.153  0.181      ·      ·      ·      ·
  4:t    0.100  0.405  0.085  0.157  0.253      ·      ·      ·
  5:␣    0.060  0.133  0.106  0.263  0.328  0.110      ·      ·
  6:C    0.099  0.055  0.175  0.200  0.165  0.142  0.164      ·
  7:i    0.151  0.094  0.123  0.072  0.142  0.148  0.150  0.120

satır toplamları: [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]

Satır t=7: 'i' token'ı (pozisyon 7) geçmişte kime ne kadar bakıyor?
  pozisyon 0 'F'    15.1%  █████████
  pozisyon 6 'C'    15.0%  █████████
  pozisyon 5 '␣'    14.8%  █████████
  pozisyon 4 't'    14.2%  █████████
  pozisyon 2 'r'    12.3%  ███████
  pozisyon 7 

In [51]:
## 4. Pozisyon embedding'i neden gerekli?
# Attention'ın kendi içinde "yer" kavramı yok (video notu 2): bir küme üzerinde çalışır.
# Dizi 0'da 'i' hem pozisyon 1'de hem pozisyon 7'de. Pozisyon embedding'i olmadan
# ikisinin key'i birebir aynıdır, bu yüzden her query onlara aynı skoru verir.         kısaca seçtiğimiz contexte aynı  karakterden iki defa varsa bunları ayırmamızı sağlıyor position embedding

# %%
print('\n=== 4. POZİSYON EMBEDDING NEDEN GEREKLİ ===')
head(tok_emb)                        # sadece token embedding, pozisyon yok
wei_pozisyonsuz = head.wei
head(x)                              # head.wei'yi pozisyonlu haline geri getir
print(f"dizi 0, satır {t}: pozisyon 1 ({etiket[1]!r}) ve pozisyon 7 ({etiket[7]!r}) aynı karakter")
print(f'  pozisyon embedding YOK : {wei_pozisyonsuz[0, t, 1].item():.4f} vs {wei_pozisyonsuz[0, t, 7].item():.4f}'
      f'  -> eşit: {torch.allclose(wei_pozisyonsuz[0, t, 1], wei_pozisyonsuz[0, t, 7])}')
print(f'  pozisyon embedding VAR : {wei[0, t, 1].item():.4f} vs {wei[0, t, 7].item():.4f}'
      f'  -> eşit: {torch.allclose(wei[0, t, 1], wei[0, t, 7])}')
print('Pozisyon olmadan model "hangi i?" sorusunu ayırt edemez; pozisyonla edebilir.')



=== 4. POZİSYON EMBEDDING NEDEN GEREKLİ ===
dizi 0, satır 7: pozisyon 1 ('i') ve pozisyon 7 ('i') aynı karakter
  pozisyon embedding YOK : 0.1608 vs 0.1608  -> eşit: True
  pozisyon embedding VAR : 0.0942 vs 0.1196  -> eşit: False
Pozisyon olmadan model "hangi i?" sorusunu ayırt edemez; pozisyonla edebilir.


In [45]:
## 5. Neden sqrt(head_size)'a bölüyoruz?
# q ve k'nın elemanları ortalaması 0, varyansı 1 ise q·k, head_size tane çarpımın toplamıdır.
# Toplamın varyansı da head_size kadar büyür. Bölme bunu 1'e geri çeker.

# %%
print('\n=== 5. NEDEN sqrt(head_size)\'A BÖLÜYORUZ ===')
print('(a) Skorların varyansı (q, k ~ N(0, 1))')
print(f'{"head_size":>10} {"bölmeden":>10} {"bölerek":>10}')
for hs in (16, 64):
    q = torch.randn(2000, T, hs)
    k = torch.randn(2000, T, hs)
    skor = q @ k.transpose(-2, -1)
    print(f'{hs:>10} {skor.var().item():>10.2f} {(skor * hs ** -0.5).var().item():>10.2f}')

# (b) Bizim head'imiz, dizi 0, satır t: softmax bölmeden vs bölerek
with torch.no_grad():
    q0 = head.query(x[0])                       # (T, hs)
    k0 = head.key(x[0])                         # (T, hs)
ham = (q0 @ k0.T)[t]                            # satır t'nin 8 skoru (son satır, maske yok)
olcekli = ham * head_size ** -0.5
p_ham, p_olcekli = F.softmax(ham, dim=-1), F.softmax(olcekli, dim=-1)
print(f'\n(b) Bizim head, dizi 0, satır {t} ({etiket[t]!r}), head_size={head_size}: softmax yan yana')
print(f'{"poz":>8} {"skor":>8} {"softmax":>9}   | {"skor/√hs":>9} {"softmax":>9}')
for s in range(T):
    print(f'{f"{s}:{etiket[s]}":>8} {ham[s].item():>8.2f} {p_ham[s].item():>9.3f}   | '
          f'{olcekli[s].item():>9.2f} {p_olcekli[s].item():>9.3f}')
print(f'{"en büyük":>8} {"":>8} {p_ham.max().item():>9.3f}   | {"":>9} {p_olcekli.max().item():>9.3f}')
assert torch.allclose(p_olcekli, wei[0, t])     # ölçekli olan, head'in kendi wei satırıyla aynı
print(f'Not: bizim q ve k varyansı 1 değil (q {q0.var().item():.2f}, k {k0.var().item():.2f}); nn.Linear\'ın varsayılan')
print('init\'i çıktıları küçük tutuyor. Bu yüzden buradaki fark (c)\'deki kadar büyük değil. q ve k büyürse')
print('ya da head_size artarsa fark büyür; (c) birim varyanslı durumu gösteriyor.')

# (c) Aynı karşılaştırma binlerce satırın ortalaması
print('\n(c) 2000 rastgele satır üzerinden (8 seçenek, düz ortalamada her biri 0.125):')
print(f'{"head_size":>10} | {"ort. en büyük ağırlık":>22} | {"tek token %90+ alıyor":>22}')
print(f'{"":>10} | {"bölmeden":>10} {"bölerek":>10} | {"bölmeden":>10} {"bölerek":>10}')
for hs in (16, 64):
    q = torch.randn(2000, T, hs)
    k = torch.randn(2000, T, hs)
    skor = (q @ k.transpose(-2, -1))[:, t]      # her örnekte son satır, 8 skor
    p1 = F.softmax(skor, dim=-1).max(-1).values
    p2 = F.softmax(skor * hs ** -0.5, dim=-1).max(-1).values
    print(f'{hs:>10} | {p1.mean().item():>10.3f} {p2.mean().item():>10.3f} | '
          f'{(p1 > 0.9).float().mean().item():>10.1%} {(p2 > 0.9).float().mean().item():>10.1%}')

# (d) videodaki mini örnek: aynı skorlar 8 ile çarpılınca
v = torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])
print('\n(d) Videodaki örnek:')
print('softmax(v)     :', [round(p, 3) for p in F.softmax(v, dim=-1).tolist()])
print('softmax(v * 8) :', [round(p, 3) for p in F.softmax(v * 8, dim=-1).tolist()], '<- one-hot\'a yaklaşıyor')

 #softmax çok büyüdüğü için değerler arasındaki fark kapanıyor ve bu da bizim sağlıklı karar almamızı engelliyor.



=== 5. NEDEN sqrt(head_size)'A BÖLÜYORUZ ===
(a) Skorların varyansı (q, k ~ N(0, 1))
 head_size   bölmeden    bölerek
        16      16.17       1.01
        64      64.19       1.00

(b) Bizim head, dizi 0, satır 7 ('i'), head_size=16: softmax yan yana
     poz     skor   softmax   |  skor/√hs   softmax
     0:F     1.95     0.209   |      0.49     0.151
     1:i     0.08     0.032   |      0.02     0.094
     2:r     1.15     0.094   |      0.29     0.123
     3:s    -1.02     0.011   |     -0.26     0.072
     4:t     1.72     0.167   |      0.43     0.142
     5:␣     1.89     0.198   |      0.47     0.148
     6:C     1.93     0.206   |      0.48     0.150
     7:i     1.03     0.083   |      0.26     0.120
en büyük              0.209   |               0.151
Not: bizim q ve k varyansı 1 değil (q 0.53, k 0.55); nn.Linear'ın varsayılan
init'i çıktıları küçük tutuyor. Bu yüzden buradaki fark (c)'deki kadar büyük değil. q ve k büyürse
ya da head_size artarsa fark büyür; (c) birim va

# **Task 4**

In [52]:
import copy
import json
import math
import os
import urllib.request

import torch
import torch.nn as nn
from torch.nn import functional as F

In [53]:
batch_size = 32
block_size = 8          # en fazla kaç karakterlik bağlam
max_iters = 5000
eval_interval = 500
eval_iters = 200
learning_rate = 1e-3    # bigram'da 1e-2 idi; self-attention yüksek lr'yi kaldırmıyor
n_embd = 32
head_size = n_embd      # tek head, boyutu n_embd (videodaki gibi)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

BIGRAM_VAL_GOREV1 = 2.4907   # Görev 1'in kaydettiği taban (bigram_baseline.json varsa oradan okunur)

# --- veri, tokenizer, split (Görev 1 ile aynı) ---
if not os.path.exists('input.txt'):
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt',
        'input.txt')
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join(itos[i] for i in l)
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]


In [54]:
def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i + block_size] for i in ix])
    y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)


@torch.no_grad()
def estimate_loss(model):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out


@torch.no_grad()
def pozisyon_bazinda_loss(model, d):
    """Split'in tamamı, örtüşmeyen block_size'lık chunk'lar halinde.
    Dönen vektörün t'inci elemanı: t+1 karakterlik bağlamla yapılan tahminlerin ortalama loss'u.
    Ortalaması, Görev 1'deki 'tüm val seti, kesin' loss ile aynı ölçü."""
    model.eval()
    n_chunks = (len(d) - 1) // block_size
    x = d[:n_chunks * block_size].view(n_chunks, block_size)
    y = d[1:n_chunks * block_size + 1].view(n_chunks, block_size)
    toplam = torch.zeros(block_size)
    for i in range(0, n_chunks, 1024):
        xb, yb = x[i:i + 1024].to(device), y[i:i + 1024].to(device)
        logits, _ = model(xb)
        l = F.cross_entropy(logits.reshape(-1, vocab_size), yb.reshape(-1), reduction='none')
        toplam += l.view(-1, block_size).sum(0).cpu()
    model.train()
    return toplam / n_chunks


In [55]:
## Model

# %%
class Head(nn.Module):
    """Tek self-attention head (Görev 3'teki)."""

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5          # head_size ile ölçekle
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        return wei @ self.value(x)


class HeadliBigram(nn.Module):
    """Bigram'ın üzerine: embedding -> (self-attention head) -> lm_head.

    head=False ile aynı model head'siz kurulur. Bu, kontrol modeli: aynı embedding'ler,
    aynı lr, aynı adım sayısı; tek fark head. Böylece düşüşün head'den geldiği ölçülebilir.
    """

    def __init__(self, head=True):
        super().__init__()
        # Bigram'da tablo (V, V) idi ve doğrudan logit veriyordu.
        # Artık token önce n_embd boyutlu bir vektöre gömülüyor, logit'leri lm_head üretiyor.
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)   # sadece 0..block_size-1
        self.sa_head = Head(head_size) if head else None
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)                                  # (B, T, C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device))  # (T, C)
        x = tok_emb + pos_emb                                                      # (B, T, C)
        if self.sa_head is not None:
            x = self.sa_head(x)                                                    # (B, T, head_size)
        logits = self.lm_head(x)                                                   # (B, T, V)
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]       # <- son block_size karaktere kırp (açıklama aşağıda)
            logits, _ = self(idx_cond)
            probs = F.softmax(logits[:, -1, :], dim=-1)
            idx = torch.cat((idx, torch.multinomial(probs, num_samples=1)), dim=1)
        return idx


def egit(model, ad):
    print(f'\n--- {ad} ---')
    print(f'{sum(p.numel() for p in model.parameters()):,} parametre')
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    for it in range(max_iters):
        if it % eval_interval == 0 or it == max_iters - 1:
            l = estimate_loss(model)
            print(f'  step {it:5d}: train {l["train"]:.4f} | val {l["val"]:.4f}')
        xb, yb = get_batch('train')
        _, loss = model(xb, yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
    return model


In [56]:
# ## Eğitim: head'li model ve head'siz kontrol modeli

# %%
print('=== EĞİTİM ===', f'({device})')
torch.manual_seed(1337)
model = egit(HeadliBigram(head=True).to(device), 'HEAD\'Lİ MODEL (tok + pos -> self-attention head -> lm_head)')
torch.manual_seed(1337)
kontrol = egit(HeadliBigram(head=False).to(device), 'KONTROL: HEAD\'SİZ (tok + pos -> lm_head), aynı ayarlar')

=== EĞİTİM === (cpu)

--- HEAD'Lİ MODEL (tok + pos -> self-attention head -> lm_head) ---
7,553 parametre
  step     0: train 4.2000 | val 4.2047
  step   500: train 2.6911 | val 2.7087
  step  1000: train 2.5196 | val 2.5303
  step  1500: train 2.4775 | val 2.4829
  step  2000: train 2.4408 | val 2.4523
  step  2500: train 2.4272 | val 2.4435
  step  3000: train 2.4130 | val 2.4327
  step  3500: train 2.3956 | val 2.4212
  step  4000: train 2.4041 | val 2.3992
  step  4500: train 2.3980 | val 2.4084
  step  4999: train 2.3951 | val 2.4126

--- KONTROL: HEAD'SİZ (tok + pos -> lm_head), aynı ayarlar ---
4,481 parametre
  step     0: train 4.4801 | val 4.4801
  step   500: train 2.7250 | val 2.7578
  step  1000: train 2.5893 | val 2.6136
  step  1500: train 2.5470 | val 2.5557
  step  2000: train 2.5184 | val 2.5313
  step  2500: train 2.5152 | val 2.5226
  step  3000: train 2.5005 | val 2.5122
  step  3500: train 2.5017 | val 2.5067
  step  4000: train 2.4970 | val 2.4914
  step  4500: 

In [57]:
## Sonuç: bigram tabanıyla yan yana

# %%
bigram_val, kaynak = BIGRAM_VAL_GOREV1, 'Görev 1 (kayıtlı değer)'
if os.path.exists('bigram_baseline.json'):
    with open('bigram_baseline.json', encoding='utf-8') as f:
        bigram_val, kaynak = json.load(f)['val_loss'], 'Görev 1 (bigram_baseline.json)'

poz_head = pozisyon_bazinda_loss(model, val_data)
poz_kontrol = pozisyon_bazinda_loss(kontrol, val_data)
head_val, kontrol_val = poz_head.mean().item(), poz_kontrol.mean().item()
head_train = pozisyon_bazinda_loss(model, train_data).mean().item()
dusus = bigram_val - head_val

print('\n=== SONUÇ: VAL LOSS (tüm val seti, kesin) ===')
print(f'{"model":<40} {"val loss":>9} {"bigrama göre":>13} {"perplexity":>11}')
print(f'{"Bigram — " + kaynak:<40} {bigram_val:>9.4f} {"":>13} {math.exp(bigram_val):>11.1f}')
kontrol_ad = "Kontrol: head'siz, aynı eğitim"
print(f'{kontrol_ad:<40} {kontrol_val:>9.4f} {kontrol_val - bigram_val:>+13.4f} {math.exp(kontrol_val):>11.1f}')
print(f'{"Tek self-attention head":<40} {head_val:>9.4f} {head_val - bigram_val:>+13.4f} {math.exp(head_val):>11.1f}')
print(f'\nDüşüş: {bigram_val:.4f} -> {head_val:.4f} = {dusus:.4f} ({dusus / bigram_val:.1%}).'
      f'  Head\'li modelin train loss\'u: {head_train:.4f}')

print('\n=== NEDEN DÜŞTÜ: POZİSYON BAZINDA VAL LOSS ===')
print('t = tahmin edilen karakterin chunk içindeki yeri; model t+1 karakter görüyor\n')
hs_ad, hl_ad = "head'siz", "head'li"
print(f'{"t":>3} {"bağlam":>7} {hs_ad:>10} {hl_ad:>9} {"fark":>8}')
for t in range(block_size):
    print(f'{t:>3} {t + 1:>5} kr {poz_kontrol[t].item():>10.4f} {poz_head[t].item():>9.4f} '
          f'{(poz_head[t] - poz_kontrol[t]).item():>+8.4f}')



=== SONUÇ: VAL LOSS (tüm val seti, kesin) ===
model                                     val loss  bigrama göre  perplexity
Bigram — Görev 1 (bigram_baseline.json)     2.4907                      12.1
Kontrol: head'siz, aynı eğitim              2.4963       +0.0056        12.1
Tek self-attention head                     2.4047       -0.0860        11.1

Düşüş: 2.4907 -> 2.4047 = 0.0860 (3.5%).  Head'li modelin train loss'u: 2.3848

=== NEDEN DÜŞTÜ: POZİSYON BAZINDA VAL LOSS ===
t = tahmin edilen karakterin chunk içindeki yeri; model t+1 karakter görüyor

  t  bağlam   head'siz   head'li     fark
  0     1 kr     2.4966    2.5553  +0.0587
  1     2 kr     2.5058    2.3986  -0.1072
  2     3 kr     2.4850    2.3691  -0.1159
  3     4 kr     2.5099    2.3890  -0.1208
  4     5 kr     2.4910    2.3784  -0.1125
  5     6 kr     2.4900    2.3770  -0.1130
  6     7 kr     2.4928    2.3814  -0.1114
  7     8 kr     2.4997    2.3887  -0.1109


In [58]:
## Üretim ve kırpma

# %%
print('\n=== ÜRETİM: neden son block_size karaktere kırpıyoruz ===')
print('pozisyon embedding tablosu:', tuple(model.position_embedding_table.weight.shape),
      f'-> sadece 0..{block_size - 1} pozisyonlarının vektörü var')
print('maske (tril):', tuple(model.sa_head.tril.shape))

uzun = data[:block_size + 1].unsqueeze(0)                  # block_size + 1 = 9 karakter
print(f'\n{block_size + 1} karakterlik girdi kırpmadan modele verilirse: {decode(uzun[0].tolist())!r}')
cpu_kopya = copy.deepcopy(model).cpu()                      # GPU'da bu hata runtime'ı bozar, CPU'da deniyoruz
try:
    cpu_kopya(uzun)
    print('  -> çalıştı (beklenmiyordu)')
except (IndexError, RuntimeError) as e:
    print(f'  -> HATA: {type(e).__name__}: {str(e).splitlines()[0]}')
print(f'Kırpınca: {decode(uzun[0, -block_size:].tolist())!r} -> çalışır, '
      f'logits {tuple(cpu_kopya(uzun[:, -block_size:])[0].shape)}')

torch.manual_seed(1337)
context = torch.zeros((1, 1), dtype=torch.long, device=device)   # 0 = '\n'
ornek = decode(model.generate(context, max_new_tokens=400)[0].tolist())
print('\nHead\'li modelden örnek metin:\n' + ornek)



=== ÜRETİM: neden son block_size karaktere kırpıyoruz ===
pozisyon embedding tablosu: (8, 32) -> sadece 0..7 pozisyonlarının vektörü var
maske (tril): (8, 8)

9 karakterlik girdi kırpmadan modele verilirse: 'First Cit'
  -> HATA: IndexError: index out of range in self
Kırpınca: 'irst Cit' -> çalışır, logits (1, 8, 65)

Head'li modelden örnek metin:

CKETYon thede.

ETI, lrolounsee weindo
Cof te thilod wist icthis ter; bea yonsenims coke ay ghare avemy mou ber sokcerl ito'ls to ghe gne ds I ary of hithe aint tisanver
We tretath; of ptis ges, lode ish rithom bie mer nof bilenaisotinoes sel nsen l'se ph Py the af, ithime:
Pr.
Srly a auce owul ote.

Y:
Y:
For
ard, I ando tong, theavithirotoull how
Hes ds win.

Bes; daipaly oleme.

TA:
AUMKAng tis


In [59]:
fark_t0 = (poz_head[0] - poz_kontrol[0]).item()
kazanc_baglam = (poz_kontrol[1:] - poz_head[1:]).mean().item()
print('\n=== AÇIKLAMA ===')
print(f"""
NE KADAR DÜŞTÜ?
Val loss bigram tabanında {bigram_val:.4f}, tek head'li modelde {head_val:.4f}: {dusus:.4f} puan
(%{dusus / bigram_val * 100:.1f}) düşüş. Perplexity {math.exp(bigram_val):.1f}'den {math.exp(head_val):.1f}'e indi; yani model her
adımda yaklaşık {math.exp(bigram_val):.0f} yerine {math.exp(head_val):.0f} eşit seçenek arasında kalmış gibi belirsiz.

NEDEN DÜŞTÜ?
Bigram bir sonraki karakteri tahmin ederken sadece son karaktere bakıyordu. Head'li modelde
her pozisyon, attention sayesinde kendinden önceki en fazla {block_size} karakterden bilgi topluyor:
query ve key'in uyumuna göre ağırlıklar hesaplanıyor, value'ların ağırlıklı ortalaması alınıyor.
Bu iki şekilde görülüyor:
1. Kontrol modeli aynı embedding'lere, aynı lr'ye ve aynı adım sayısına sahip, sadece head'i yok.
   Onun val loss'u {kontrol_val:.4f}, bigram'la neredeyse aynı. Yani düşüş daha uzun eğitimden ya da
   embedding katmanından değil, head'den geliyor.
2. Pozisyon bazındaki tabloda t=0'da model sadece tek karakter görüyor, toplayacak geçmiş yok.
   Orada head bir şey kazandırmıyor, hatta head'li model biraz daha kötü ({fark_t0:+.4f}); muhtemelen
   katmanları, geçmişin karıştığı diğer pozisyonlara göre ayarlandığı için. Bağlamın olduğu
   pozisyonlarda (t=1..{block_size - 1}) ise head'li model ortalama {kazanc_baglam:.4f} daha iyi; head'siz model
   bağlam uzasa da yerinde sayıyor. Yani kazancın tamamı geçmişten toplanan bilgiden geliyor.

Düşüş neden daha büyük değil? Tek bir head var ve {head_size} boyutlu. Bağlam en fazla {block_size} karakter.
Token'lar bilgiyi topluyor ama ardından bu bilgiyi işleyecek bir katman (feedforward) yok. Ayrıca
token'ın kendi kimliği de sadece attention'ın çıktısı üzerinden lm_head'e ulaşıyor. Videonun
sonraki adımları (multi-head, feedforward, residual) tam olarak bunları ekliyor.

ÜRETİRKEN NEDEN SON {block_size} KARAKTERE KIRPIYORUZ?
generate her adımda ürettiği karakteri girdinin sonuna ekliyor, girdi her adımda bir uzuyor.
Ama pozisyon embedding tablosunun sadece {block_size} satırı var (pozisyon 0..{block_size - 1}). {block_size + 1}. karakter
için bir pozisyon vektörü yok; yukarıda görüldüğü gibi model hata veriyor. Maske de {block_size}x{block_size}.
Ayrıca model eğitimde hiçbir zaman {block_size}'den uzun bir bağlam görmedi. Bu yüzden her adımda sadece
son {block_size} karakteri veriyoruz: kayan bir pencere. Daha eskisi unutuluyor; modelin hafızası
block_size kadar. Bigram'da bu gerekmiyordu, çünkü pozisyon embedding'i yoktu ve zaten sadece son
karaktere bakıyordu.
""")

with open('tek_head_sonuc.json', 'w', encoding='utf-8') as f:
    json.dump({'model': 'bigram + tek self-attention head', 'val_loss': round(head_val, 4),
               'train_loss': round(head_train, 4), 'kontrol_headsiz_val_loss': round(kontrol_val, 4),
               'bigram_val_loss': bigram_val, 'pozisyon_bazinda_val_loss': [round(v, 4) for v in poz_head.tolist()],
               'ornek_metin': ornek, 'device': device}, f, ensure_ascii=False, indent=2)
print('kaydedildi: tek_head_sonuc.json')



=== AÇIKLAMA ===

NE KADAR DÜŞTÜ?
Val loss bigram tabanında 2.4907, tek head'li modelde 2.4047: 0.0860 puan
(%3.5) düşüş. Perplexity 12.1'den 11.1'e indi; yani model her
adımda yaklaşık 12 yerine 11 eşit seçenek arasında kalmış gibi belirsiz.
 
NEDEN DÜŞTÜ?
Bigram bir sonraki karakteri tahmin ederken sadece son karaktere bakıyordu. Head'li modelde
her pozisyon, attention sayesinde kendinden önceki en fazla 8 karakterden bilgi topluyor:
query ve key'in uyumuna göre ağırlıklar hesaplanıyor, value'ların ağırlıklı ortalaması alınıyor.
Bu iki şekilde görülüyor:
1. Kontrol modeli aynı embedding'lere, aynı lr'ye ve aynı adım sayısına sahip, sadece head'i yok.
   Onun val loss'u 2.4963, bigram'la neredeyse aynı. Yani düşüş daha uzun eğitimden ya da
   embedding katmanından değil, head'den geliyor.
2. Pozisyon bazındaki tabloda t=0'da model sadece tek karakter görüyor, toplayacak geçmiş yok.
   Orada head bir şey kazandırmıyor, hatta head'li model biraz daha kötü (+0.0587); muhtemelen
   katma